# Stage 5: the analysis year

**Reads:** `data/gravity_clean.csv`, saved by stage 2, and `results/size_rule.csv`, saved by stage 4 and frozen after it (decision 21).

**Writes:**
- `results/analysis_runs.csv`: one row per method and training draw in the analysis year, with its test deviance, the other scores, its seconds and its number of fits.
- `results/analysis_summary.csv`: one row per training size and method, with the mean, standard error and median of the test deviance, the seconds and fits, and the comparisons of the plan: the draws in which `ppml` wins, `size_rule` against `cv_select` (the test of H2) and against `best_of_three`, and `ridge` against `ppml`.
- `results/analysis_cutoff.csv`: the pattern of the gap between `ppml` and the better of `rf` and `gbm` over the sizes, with its bootstrap (the test of H1), and the same for `ridge` (decision 22, beside the test of H1 and not part of it).
- `results/selection_confusion.csv`: the model each selector picked against the model that won, per size and over all sizes.

This is the first stage that reads the analysis year. Nothing is tuned here: `ppml`, `rf` and `gbm` are fitted once per draw at the settings `size_rule.csv` gives for the size, and the only choices made on a training sample are those of `cv_select` and of the ridge line, each by cross-validation inside the sample. The test set is never used to fit a model or to choose one, except by `best_of_three`, a reference that is not a selector. The steps are those of stage 5 in section 6 of `docs/build_plan.md`.

Every standard error and interval here describes variation over training draws from one pool, scored on one test set. The draws overlap, more so at large sizes.

*Drafted with Claude (Anthropic), which was asked for stage 5 of the build plan after the freeze (decision 21): the analysis year at the frozen settings, cross-validated selection, the ridge line, the summary with the test of H2, the cutoff of H1 with its bootstrap and the cutoff test of `ridge` (decision 22), and the selection table. It replaces the placeholder of 18 October 2026.*

In [ ]:
from helpers import (ANALYSIS_YEAR, BOOTSTRAP, CV_DRAWS, CV_SELECT_FITS, DATA, DRAWS, N_JOBS, PATTERNS, PREDICTORS,
                     QUICK, RESULTS, RIDGE_ALPHAS, RIDGE_FITS, ROOT, SIZES, TARGET, analysis_crossings,
                     analysis_cutoff_table, analysis_matrices, analysis_rows, analysis_runs_table, analysis_summary,
                     bootstrap_analysis, candidate_settings, check_analysis_gaps, check_find_crossing, frozen_settings,
                     projected_seconds, refuse_part, run_analysis_draw, run_analysis_draws, selection_confusion_table,
                     size_rule_lookup, split_pairs)

import hashlib
import time

import pandas as pd

refuse_part()
# Section 8: find_crossing on toy gaps, before it is used on real ones.
check_find_crossing()

# The analysis year is split once, with split 0, as the development year was.
SPLIT = 0
# Decision 21: the SHA-256 of results/size_rule.csv as stage 4 wrote it in commit 6c33375, the file Yu froze.
FROZEN_SIZE_RULE_SHA256 = "fd6b55911f43d0b461bbac34762dbbd61ea51b1c57fd055d6f4a5c3bdffc4191"
assert CV_DRAWS <= DRAWS, "CV_DRAWS is larger than DRAWS."
print(f"Analysis year {ANALYSIS_YEAR}; sizes {list(SIZES)}; {DRAWS} draws per size, cv_select on the first {CV_DRAWS}; "
      f"{N_JOBS} worker(s)")

## The frozen settings

`size_rule.csv` gives, at each size, the setting of `ppml`, tuned `rf` and tuned `gbm`, and the model the size rule chooses. In the full run, the file must be the one frozen after stage 4 (decision 21, commit `6c33375`): the stage stops if its SHA-256 differs. Quick mode reads the file of the quick run of stage 4 instead. The settings at a size come from `size_rule_lookup`, the size of the file nearest to it on a log scale, which is the size itself, since the file has every size of `SIZES`.

In [ ]:
rule_path = RESULTS / "size_rule.csv"
with open(rule_path, "rb") as file:
    rule_sha256 = hashlib.file_digest(file, "sha256").hexdigest()
print("SHA-256 of size_rule.csv:", rule_sha256)
if not QUICK:
    assert rule_sha256 == FROZEN_SIZE_RULE_SHA256, \
        "size_rule.csv is not the file frozen in decision 21 (commit 6c33375). Stop and report."
    print("It is the file frozen in decision 21.")
# round_trip reads every number back exactly as stage 4 wrote it.
rule = pd.read_csv(rule_path, float_precision="round_trip")
assert sorted(int(n) for n in rule["size"].unique()) == list(SIZES), \
    f"size_rule.csv has the sizes {sorted(rule['size'].unique())}, not SIZES {list(SIZES)}."
settings_at = {n: frozen_settings(size_rule_lookup(rule, n)) for n in SIZES}
for n in SIZES:
    assert set(size_rule_lookup(rule, n)["model"]) == {"ppml", "rf", "gbm"}, f"size_rule.csv lacks a model at size {n}."
    settings = settings_at[n]
    print(f"n = {n:>5,}: rf {settings['rf']['config']}, gbm {settings['gbm']['config']}; "
          f"the size rule chooses {settings['chosen']}")

## Split the analysis year into a training pool and a test set

As in stage 3: a random quarter of the unordered country pairs goes to the test set with all its rows (`split_pairs`), so both directions of a pair land on the same side. The largest size has to be at most a quarter of the pool. If it is not, the stage stops here: the size is not shrunk silently.

In [ ]:
# Country identifiers are text, as in stage 2: no value is read as missing.
table = pd.read_csv(DATA / "gravity_clean.csv", dtype={"country_id_o": str, "country_id_d": str, "pair_id": str},
                    keep_default_na=False, na_values=[""])
pool, test = split_pairs(table, ANALYSIS_YEAR, SPLIT)
print(f"{ANALYSIS_YEAR}: {(table['year'] == ANALYSIS_YEAR).sum():,} rows")
print(f"  training pool: {len(pool):,} rows, {pool['pair_id'].nunique():,} pairs")
print(f"  test set:      {len(test):,} rows, {test['pair_id'].nunique():,} pairs")

X_pool = pool[PREDICTORS].to_numpy(dtype=float)
y_pool = pool[TARGET].to_numpy(dtype=float)
# The folds of cv_select and of the ridge line keep both directions of a pair together.
pair_pool = pool["pair_id"].to_numpy()
X_test = test[PREDICTORS].to_numpy(dtype=float)
y_test = test[TARGET].to_numpy(dtype=float)

largest = max(SIZES)
assert largest <= len(pool) / 4, f"The largest size, {largest:,}, is over a quarter of the pool ({len(pool):,} rows). Stop and report."
print(f"Largest size {largest:,}: {largest / len(pool):.1%} of the pool")

## Time one draw

One draw at the largest size is timed, with `cv_select`, and the minutes of the stage are projected from it with the ratios between sizes measured in section 9 of the plan: `DRAWS` draws per size, `CV_DRAWS` of them with `cv_select`. Only seconds are printed here, no error. The notebook does not wait: before the full run in Nuvolos, Claude projects the time from the seconds of the full run of stage 3, and asks Yu about `CV_DRAWS` (decision 4) if it is long, so that this number is fixed before any error of this stage is looked at.

In [ ]:
start = time.perf_counter()
timed = run_analysis_draw(X_pool, y_pool, pair_pool, X_test, y_test, ANALYSIS_YEAR, SPLIT, largest, 0,
                          settings_at[largest], 1)
with_cv = time.perf_counter() - start
without_cv = with_cv - timed["cv_select"]["seconds"]
projected = sum(CV_DRAWS * projected_seconds(with_cv, largest, n)
                + (DRAWS - CV_DRAWS) * projected_seconds(without_cv, largest, n) for n in SIZES) / 60
print(f"One draw at n = {largest:,}: {with_cv:.1f} seconds, {timed['cv_select']['seconds']:.1f} of them in the "
      f"{timed['cv_select']['fits']} fits of cv_select")
print(f"Projected time of stage 5: {projected:.1f} minutes on one worker"
      + (f", about {projected / N_JOBS:.1f} with {N_JOBS} workers" if N_JOBS > 1 else ""))

## Steps 1 to 5. The draws

For every size and every draw (`run_analysis_draw` in `helpers.py`), on one training sample of n pool rows drawn without replacement:

1. `ppml`, `rf` and `gbm` are fitted once each at the settings frozen for that size (`fit_setting`). `gbm` is fitted with its tuned number of trees, and `rf` without out-of-bag predictions, which stage 5 does not use. Each is one fit.
2. `size_rule` is the row of the model the size rule chooses at that size, copied. Its cost is that one fit.
3. `best_of_three` is the row of whichever of the three has the lowest test deviance on this draw (a tie goes to the first of `ppml`, `rf`, `gbm`). It looks at the test set, so it is a reference and not a selector. Its cost is the three fits.
4. `cv_select`, on the first `CV_DRAWS` draws: 5-fold cross-validation on the training sample over every candidate configuration (`cv_select`). The folds keep both directions of a pair together and are the same folds as those of the ridge line. On each fold, every configuration is fitted on the other four folds and scored by its mean Poisson deviance on the fold: one `ppml` fit, one fit for each `rf` setting, and one `gbm` fit per depth scored at every number of trees through staged prediction. The configuration with the lowest mean over the five folds is refitted on the whole sample and scored on the test set; a tie goes to the simpler setting within a model (as in the tuning of stage 4) and to the first of `ppml`, `rf`, `gbm` across models. Its cost is every fit from the first fold to the refit.
5. `ridge`, the ridge line (decisions 15 and 20): its penalty chosen by 5-fold cross-validation inside the sample, then one refit (`fit_ridge`, as in stage 3). A reference line: no selector sees it.

Every prediction is checked to be positive and finite, and each method gives one row of `analysis_runs.csv`: test deviance, D squared, MAE and RMSE each divided by the mean test flow, the seconds of its fit calls and its number of fits. After each size, the notebook prints the checks on `ppml` of section 8 (fits that fell back to L-BFGS, draws with an indicator constant in the sample, and the largest relative difference between the predicted and the observed training total) and, for each penalty of the grid, in how many draws the ridge line chose it.

In [ ]:
print(f"Fits per draw: ppml, rf, gbm and size_rule 1 each, best_of_three 3, cv_select {CV_SELECT_FITS} "
      f"(over {len(candidate_settings())} candidate configurations), ridge {RIDGE_FITS}")
runs = []
for n in SIZES:
    start = time.perf_counter()
    size_rows = []
    for draw, result in enumerate(run_analysis_draws(X_pool, y_pool, pair_pool, X_test, y_test, ANALYSIS_YEAR, SPLIT,
                                                     n, DRAWS, settings_at[n], CV_DRAWS)):
        keys = {"year": ANALYSIS_YEAR, "split": SPLIT, "size": n, "draw": draw}
        size_rows.extend(analysis_rows(result, y_test, settings_at[n], keys))
    size_runs = analysis_runs_table(size_rows)
    runs.append(size_runs)

    ppml = size_runs[size_runs["method"] == "ppml"]
    print(f"n = {n:>5,}: {DRAWS} draws in {(time.perf_counter() - start) / 60:.1f} minutes. "
          f"ppml: {int(ppml['lbfgs_fallback'].sum())} fit(s) fell back to L-BFGS, "
          f"{int((ppml['constant_columns'] > 0).sum())} draw(s) with a constant indicator, "
          f"largest relative difference of the training total {(ppml['train_total_ratio'] - 1).abs().max():.1e}")
    chosen = size_runs.loc[size_runs["method"] == "ridge", "ridge_alpha"]
    print("           ridge: draws in which cross-validation chose each penalty:")
    for alpha in RIDGE_ALPHAS:
        print(f"             {alpha:>7}: {int((chosen == alpha).sum())}")

runs = pd.concat(runs, ignore_index=True)
print(f"analysis_runs.csv: {len(runs):,} rows")

## The summary and the test of H2

`analysis_summary.csv` has one row per size and method: the mean, standard error (standard deviation over draws divided by the square root of the number of draws) and median of the test deviance, whether one draw holds more than half of the sum over draws (`se_unreliable`), and the mean seconds and fits per draw. Then, on one method's rows each:

- `ppml`: which of `rf` and `gbm` has the lower mean at that size (`best`; a tie goes to `rf`), and the number of draws in which `ppml` has the lower test deviance than `rf`, `gbm` and `best` on the same training sample (decision 12; a tie is not a win).
- `size_rule`: the mean paired difference `size_rule` minus `cv_select` over the draws that have both, its standard error, and whether the difference is at most twice its standard error (`within_2se`). H2 says this holds at every size, at lower fitting cost. Beside it, the paired difference `size_rule` minus `best_of_three`.
- `ridge`: the paired difference `ridge` minus `ppml` with its standard error, and the draws in which `ridge` beats `best` (decisions 20 and 22), outside H1 and H2.

In [ ]:
summary = analysis_summary(runs)
with pd.option_context("display.width", 200, "display.max_columns", None):
    print(summary[["size", "method", "config", "draws", "mean", "se", "se_unreliable", "median", "seconds", "fits"]]
          .to_string(index=False, float_format="{:.4g}".format))
    print()
    print("The test of H2, size_rule against cv_select (and against best_of_three):")
    h2 = summary[summary["method"] == "size_rule"].set_index("size")
    cv = summary[summary["method"] == "cv_select"].set_index("size")
    h2 = h2[["minus_cv_select", "se_minus_cv_select", "draws_with_cv_select", "within_2se",
             "minus_best_of_three", "se_minus_best_of_three"]].assign(
        seconds_size_rule=h2["seconds"], seconds_cv_select=cv["seconds"], fits_size_rule=h2["fits"],
        fits_cv_select=cv["fits"])
    print(h2.to_string(float_format="{:.4g}".format))
    print(f"within_2se at {int(h2['within_2se'].sum())} of {len(h2)} sizes")
    print()
    print("Draws in which ppml beats each ensemble, and ridge against ppml and the better ensemble:")
    ppml_rows = summary[summary["method"] == "ppml"].set_index("size")
    ridge_rows = summary[summary["method"] == "ridge"].set_index("size")
    print(pd.concat([ppml_rows[["draws", "best", "ppml_beats_rf", "ppml_beats_gbm", "ppml_beats_best"]],
                     ridge_rows[["minus_ppml", "se_minus_ppml", "ridge_beats_best"]]], axis=1)
          .to_string(float_format="{:.4g}".format))

## The cutoff: the test of H1, and `ridge` beside it

`find_crossing` (section 7) reads, over the sizes, the gap E(ppml) minus the lower of E(rf) and E(gbm) at the frozen settings, in the row `best`. This is the test of H1, read with decision 12: H1 is supported only on `one_crossing`; on `reversal_then_crossing` the paper reports `last_crossing` and the reversal below it. The row `ridge_best` does the same for E(ridge) minus the lower of E(rf) and E(gbm) (decision 22): it is reported beside the test of H1 and is not part of it. For the patterns `reversal` and `reversal_then_crossing`, `reversal_from` and `reversal_to` are the two sizes around the change where `ppml` (or `ridge`) goes from behind or level to ahead, the gap from zero or positive to negative.

The interval comes from the bootstrap of stage 4 step 6, with nothing tuned: the draws are resampled with replacement `BOOTSTRAP` times, separately at each size, with the same resampled draws for every model and for both rows (resample b at size n uses `rng("bootstrap", year, split, n, b)`); inside each resample the means, the better ensemble and both gaps are recomputed and `find_crossing` read again. First, the same code on every draw once must give the gaps of the summary, to a relative 1e-9 of the means.

In [ ]:
matrices = analysis_matrices(runs)
point = check_analysis_gaps(matrices, summary)
print("On every draw once, the bootstrap gives the better ensemble and the gaps of the summary to a relative 1e-9 of the means.")
print(point.to_string(index=False, float_format="{:.4g}".format))
crossing_table = analysis_crossings(point)
resampled = bootstrap_analysis(matrices, ANALYSIS_YEAR, SPLIT, BOOTSTRAP)
cutoff = analysis_cutoff_table(crossing_table, resampled)
print()
print(f"{BOOTSTRAP:,} resamples")
print(cutoff[["comparison", "pattern", "n_star", "last_crossing", "crossing_from", "crossing_to", "reversal_from",
              "reversal_to"]].to_string(index=False, float_format="{:.1f}".format))
print()
print("Share of resamples in each pattern:")
print(cutoff[["comparison"] + [f"share_{pattern}" for pattern in PATTERNS]].to_string(index=False, float_format="{:.3f}".format))
print()
print(cutoff[["comparison", "resamples_one_crossing", "n_star_low", "n_star_high", "resamples_reversal_then_crossing",
              "last_crossing_low", "last_crossing_high"]].to_string(index=False, float_format="{:.1f}".format))

## The selection table

On the draws that have both selectors (the first `CV_DRAWS` of each size): the model each selector picked against the model `best_of_three` found best on that draw (a tie goes to the first of `ppml`, `rf`, `gbm`). For `cv_select` the model picked is the model of its chosen configuration, whatever its setting. The size rule picks one model per size, so at one size its table is one row. For each selector and scope (one size, or all sizes pooled), the row `total` gives the share picked right, the share that always picking the most frequent winner of each size would get (no selector that picks one model per size can do better), and the mean and paired standard error of the selector's test deviance minus that of `best_of_three`. For `cv_select` this difference can be negative, since its configuration need not be one of the three frozen settings, so it is called the difference, not the regret. The share picked right is descriptive: H2 is read on the deviance alone.

In [ ]:
confusion = selection_confusion_table(runs)
with pd.option_context("display.width", 200, "display.max_columns", None):
    print(confusion.to_string(index=False, float_format="{:.4g}".format))

## Write the four files

In [ ]:
RESULTS.mkdir(exist_ok=True)
for table, name in [(runs, "analysis_runs.csv"), (summary, "analysis_summary.csv"), (cutoff, "analysis_cutoff.csv"),
                    (confusion, "selection_confusion.csv")]:
    table.to_csv(RESULTS / name, index=False, lineterminator="\n")
    print(f"Wrote {(RESULTS / name).relative_to(ROOT)}: {len(table):,} rows")